# XGBoost: ¿Cuánto vale una casa en California?

Gradient boosting aplicado a precios de vivienda. Comparación Linear Regression → Random Forest → XGBoost.
Hyperparameter tuning con GridSearchCV, learning curves y SHAP para explainability.

**Dataset:** [California Housing](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) (sklearn built-in)  
**Autor:** @aroaxinping  
**Fecha:** Abril 2026

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

ACCENT  = '#e85d04'
ACCENT2 = '#6a9ad4'
ACCENT3 = '#2dc653'
WARN    = '#f4d03f'

print('Entorno listo.')

---
## 1. Datos

| Dataset | Fuente | Registros | Target |
|---|---|---|---|
| California Housing | sklearn built-in | 20,640 bloques censales | Precio medio ($100k) |

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

DATA_PATH = Path('../data/processed/housing_clean.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    es_sintetico = False
    print(f'[OK] Datos cargados: {len(df)} bloques censales')
else:
    print('[INFO] Datos no encontrados. Generando dataset sintético...')
    print('[TIP]  Ejecuta: python src/fetch_housing.py')
    from fetch_housing import generate_synthetic_housing
    df = generate_synthetic_housing()
    df['RoomsPerPerson'] = df['AveRooms'] / df['AveOccup'].clip(lower=0.1)
    df['BedroomRatio'] = df['AveBedrms'] / df['AveRooms'].clip(lower=0.1)
    es_sintetico = True

if es_sintetico:
    print('\n⚠️  AVISO: Datos sintéticos.')

TARGET = 'MedHouseVal'
FEATURES = [c for c in df.columns if c != TARGET]

print(f'\nFeatures ({len(FEATURES)}): {FEATURES}')
print(f'Target: {TARGET} (precio medio en $100k)')
print(f'\nTarget stats:')
print(df[TARGET].describe().round(3))
df.head()

---
## 2. Exploración rápida

> **Pregunta:** ¿Qué features tienen más relación con el precio de la vivienda?

In [ ]:
# 2.1 Correlaciones con el target
corr_target = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = [ACCENT if c > 0 else ACCENT2 for c in corr_target]
ax.barh(corr_target.index, corr_target.values, color=colors, edgecolor='#333')
ax.axvline(0, color='#555', lw=0.8)
ax.set(xlabel='Correlación con precio', title='Features vs MedHouseVal')
plt.tight_layout()
plt.show()

print('Top feature:', corr_target.index[0], f'(r = {corr_target.iloc[0]:.3f})')

In [ ]:
# 2.2 Distribución del target
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df[TARGET], bins=50, color=ACCENT, alpha=0.85, edgecolor='#333')
ax.axvline(df[TARGET].median(), color=ACCENT2, ls='--', lw=1.5, label=f'mediana = ${df[TARGET].median()*100:.0f}k')
ax.set(xlabel='Precio Medio ($100k)', ylabel='Frecuencia', title='Distribución de Precios')
ax.legend()
plt.tight_layout()
plt.show()

# Nota sobre el cap en $500k
capped = (df[TARGET] >= 5.0).mean() * 100
print(f'Muestras en el tope ($500k+): {capped:.1f}% — el target está censurado.')

In [ ]:
# 2.3 Train/test split
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalar para Linear Regression
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=FEATURES, index=X_test.index)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

---
## 3. Baseline: Linear Regression + Random Forest

> **Pregunta:** ¿Qué rendimiento dan los modelos que ya conocemos antes de probar XGBoost?

In [ ]:
# 3.1 Linear Regression
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

r2_lr = r2_score(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print(f'--- Linear Regression ---')
print(f'  R²:   {r2_lr:.4f}')
print(f'  RMSE: {rmse_lr:.4f} ($100k)')
print(f'  MAE:  {mae_lr:.4f} ($100k)')

In [ ]:
# 3.2 Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f'--- Random Forest ---')
print(f'  R²:   {r2_rf:.4f}')
print(f'  RMSE: {rmse_rf:.4f}')
print(f'  MAE:  {mae_rf:.4f}')

---
## 4. XGBoost (default)

> **Pregunta:** ¿XGBoost con parámetros por defecto ya supera a Random Forest?

In [ ]:
# 4.1 XGBoost default
xgb_default = xgb.XGBRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_default.fit(X_train, y_train)
y_pred_xgb = xgb_default.predict(X_test)

r2_xgb = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

print(f'--- XGBoost (default) ---')
print(f'  R²:   {r2_xgb:.4f}')
print(f'  RMSE: {rmse_xgb:.4f}')
print(f'  MAE:  {mae_xgb:.4f}')

---
## 5. Hyperparameter Tuning

> **Pregunta:** ¿Cuánto mejora XGBoost si tuneamos learning_rate, max_depth y n_estimators?

In [ ]:
# 5.1 GridSearchCV
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

print(f'Combinaciones totales: {np.prod([len(v) for v in param_grid.values()])}')
print('Esto puede tardar unos minutos...\n')

grid = GridSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=0,
)
grid.fit(X_train, y_train)

print('--- Mejores hiperparámetros ---')
for k, v in grid.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor RMSE (CV): {-grid.best_score_:.4f}')

In [ ]:
# 5.2 Evaluar mejor modelo
xgb_tuned = grid.best_estimator_
y_pred_tuned = xgb_tuned.predict(X_test)

r2_tuned = r2_score(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)

print(f'--- XGBoost (tuned) ---')
print(f'  R²:   {r2_tuned:.4f}')
print(f'  RMSE: {rmse_tuned:.4f}')
print(f'  MAE:  {mae_tuned:.4f}')
print(f'\nMejora RMSE vs default: {(rmse_xgb - rmse_tuned) / rmse_xgb * 100:+.1f}%')

---
## 6. Learning Curves

> **Pregunta:** ¿El modelo necesita más datos o más complejidad?

In [ ]:
# 6.1 Learning curves del modelo tuneado
train_sizes, train_scores, val_scores = learning_curve(
    xgb_tuned, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=3, scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

train_rmse = -train_scores.mean(axis=1)
val_rmse = -val_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_rmse, color=ACCENT, lw=2, marker='o', ms=5, label='Train RMSE')
ax.plot(train_sizes, val_rmse, color=ACCENT2, lw=2, marker='s', ms=5, label='Validation RMSE')
ax.fill_between(train_sizes, train_rmse - train_std, train_rmse + train_std, alpha=0.1, color=ACCENT)
ax.fill_between(train_sizes, val_rmse - val_std, val_rmse + val_std, alpha=0.1, color=ACCENT2)
ax.set(xlabel='Training Set Size', ylabel='RMSE', title='Learning Curves — XGBoost Tuned')
ax.legend()
plt.tight_layout()
plt.show()

gap = val_rmse[-1] - train_rmse[-1]
print(f'Gap final train-val: {gap:.4f}')
if gap > 0.1:
    print('→ Alto gap: el modelo podría beneficiarse de más datos o regularización.')
else:
    print('→ Bajo gap: buena generalización con los datos actuales.')

---
## 7. SHAP: Explainability

> **Pregunta:** ¿Por qué el modelo predice precios altos o bajos? ¿Qué features pesan más?

In [ ]:
# 7.1 SHAP values
try:
    import shap
    
    explainer = shap.TreeExplainer(xgb_tuned)
    shap_values = explainer.shap_values(X_test)
    
    # Summary plot
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
    plt.title('SHAP Feature Importance', color='#fff', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Beeswarm
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, show=False)
    plt.title('SHAP Beeswarm — Impacto de Cada Feature', color='#fff', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print('SHAP explica cómo cada feature contribuye a cada predicción individual.')
    print('Rojo = valor alto de la feature, Azul = valor bajo.')
    
except ImportError:
    print('[WARN] pip install shap para ver los SHAP plots.')
    print('Usando feature_importances_ de XGBoost como fallback...')
    
    fi = pd.Series(xgb_tuned.feature_importances_, index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(fi.index, fi.values, color=ACCENT, edgecolor='#333')
    ax.set(xlabel='Feature Importance (gain)', title='XGBoost Feature Importance')
    plt.tight_layout()
    plt.show()

---
## 8. Predicted vs Actual

In [ ]:
# 8.1 Comparación visual de los 4 modelos
models = [
    ('Linear Regression', y_pred_lr, r2_lr),
    ('Random Forest', y_pred_rf, r2_rf),
    ('XGBoost (default)', y_pred_xgb, r2_xgb),
    ('XGBoost (tuned)', y_pred_tuned, r2_tuned),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, y_pred, r2) in zip(axes, models):
    ax.scatter(y_test, y_pred, alpha=0.15, s=5, color=ACCENT)
    lims = [0, 5.5]
    ax.plot(lims, lims, '--', color=ACCENT3, lw=1.5)
    ax.set(xlabel='Real', ylabel='Predicho', title=f'{name}\nR² = {r2:.3f}', xlim=lims, ylim=lims)
    ax.set_aspect('equal')

plt.suptitle('Predicted vs Actual — 4 Modelos', y=1.02, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Síntesis y conclusiones

In [ ]:
# 9.1 Tabla resumen
resumen = pd.DataFrame([
    {'Modelo': 'Linear Regression', 'R²': f'{r2_lr:.4f}', 'RMSE': f'{rmse_lr:.4f}', 'MAE': f'{mae_lr:.4f}'},
    {'Modelo': 'Random Forest', 'R²': f'{r2_rf:.4f}', 'RMSE': f'{rmse_rf:.4f}', 'MAE': f'{mae_rf:.4f}'},
    {'Modelo': 'XGBoost (default)', 'R²': f'{r2_xgb:.4f}', 'RMSE': f'{rmse_xgb:.4f}', 'MAE': f'{mae_xgb:.4f}'},
    {'Modelo': 'XGBoost (tuned)', 'R²': f'{r2_tuned:.4f}', 'RMSE': f'{rmse_tuned:.4f}', 'MAE': f'{mae_tuned:.4f}'},
])
print(resumen.to_markdown(index=False))

mejora_total = (rmse_lr - rmse_tuned) / rmse_lr * 100
print(f'\nMejora total RMSE (Linear → XGBoost tuned): {mejora_total:.1f}%')
print(f'\n--- Conclusión ---')
print(f'XGBoost tuneado es el mejor modelo con R² = {r2_tuned:.3f}.')
print(f'MedInc (ingreso medio) domina todas las predicciones — SHAP lo confirma.')
print(f'El hyperparameter tuning aporta una mejora modesta pero consistente.')
print(f'La ubicación (lat/lon) captura diferencias de precio entre zonas.')
print(f'\nPróximo paso: aplicar todo esto a un predictor de viralidad con datos propios.')